# Phase 6d: controls orthogonal to v (Qwen2.5-7B-Instruct, bf16)

**A100**, then `Runtime → Run all`. Needs the Phase 5/6/6b/6c outputs on Drive.

Phase 6c: ablating v at every layer from the pushback turn erased the senior − junior gap (−5.9 points), more than any of
17 controls, but two shuffled vectors got to −4.2. Ablation ignores sign, so a shuffled vector that still overlaps v
(|cos| > 0) partly removes v too. This run repeats the 7 shuffled controls **with their v component projected out**
(same length, data-shaped, cos(control, v) = 0), at all layers and in layers 10–19. The report also prints each
control's cos² with v, so the existing units can be checked too.

- orthogonal controls ≈ 0 → the gap really runs through v
- orthogonal controls still erase the gap → senior/junior information is spread over many directions

Resumable: if Colab drops, `Run all` again.

In [ ]:
CFG = "configs/colab.yaml"
AUTO_DISCONNECT = True
BS = 64

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None
url = f"https://{token}@github.com/GTRAN7/vnpronouns.git" if token else "https://github.com/GTRAN7/vnpronouns.git"
if not os.path.exists('/content/vnpronouns'):
    subprocess.run(['git', 'clone', '-q', url, '/content/vnpronouns'], check=True)
else:
    subprocess.run(['git', '-C', '/content/vnpronouns', 'pull', '-q'], check=True)
%cd /content/vnpronouns
!git log --oneline -1
!pip install -q -e .

def run(cmd, log, env=None):
    """Run a shell command, stream + log its output, and raise if it fails (so Run all stops).
    Splits only on real newlines and keeps the text after the last carriage return, so tqdm
    progress bars collapse to one line instead of flooding the notebook."""
    print(f'$ {cmd}', flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1', **(env or {})})
    with open(log, 'a', encoding='utf-8') as f:
        for raw in p.stdout:
            line = raw.decode('utf-8', 'replace').rstrip('\n').split('\r')[-1]
            if line.strip():
                print(line, flush=True)
                f.write(line + '\n')
    if p.wait() != 0:
        raise RuntimeError(f'FAILED (exit {p.returncode}): {cmd}  -- see {log}')

## 1. GPU check + smoke test

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> A100 (or L4)'
name = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f'{name}, {mem:.0f} GB, bf16 supported: {torch.cuda.is_bf16_supported()}')
assert torch.cuda.is_bf16_supported() and mem >= 22, f'{name} cannot hold the 7B in bf16; switch to A100/L4'

import os, shutil, time
OUT = '/content/drive/MyDrive/vnpronouns/outputs/Qwen2.5-7B-Instruct'
assert os.path.exists(f'{OUT}/vectors.pt'), 'vectors.pt missing: run colab_7b_direction.ipynb first'
assert os.path.exists(f'{OUT}/steer/final_ablate/none_L-1_lam+0.csv'), 'Phase 6b/6c units missing'
SMOKE = {'VNP_OUTPUT_DIR': '/content/smoke'}
shutil.rmtree('/content/smoke', ignore_errors=True)
t0 = time.time()
L = '/content/log_smoke.txt'
run(f'python -m vnpronouns.run_behavior --config {CFG} --pool 2 --no-filter', L, SMOKE)
run(f'python -m vnpronouns.extract --config {CFG} --acts-path /content/smoke/acts.pt', L, SMOKE)
run(f'python -m vnpronouns.steer --config {CFG} --batch-size {BS} --split test --layers 8 --lams 1 --vectors deference', L, SMOKE)
run(f'python -m vnpronouns.steer --config {CFG} --batch-size {BS} --split test --mode ablate --layers all 10-19 --vectors deference shuffled --pair-shuffled --orth', L, SMOKE)
run(f'python -m vnpronouns.steer_report --config {CFG} --split test --section ablate --n-boot 200', L, SMOKE)
print(f'\nSMOKE TEST PASSED in {time.time() - t0:.0f}s')
shutil.rmtree('/content/smoke', ignore_errors=True)

## 2. Orthogonalized shuffled controls (all layers and layers 10–19, from the pushback turn)

In [ ]:
L = '/content/log_orth.txt'
run(f'python -m vnpronouns.steer --config {CFG} --batch-size {BS} --split final --mode ablate --layers all 10-19 --vectors shuffled --pair-shuffled --orth', L)
!rm -f /content/log_ablate_report.txt
run(f'python -m vnpronouns.steer_report --config {CFG} --split final --section ablate', '/content/log_ablate_report.txt')

## 3. Summary to paste back to Claude

In [ ]:
import re, torch
def clean(path):
    lines = open(path, encoding='utf-8').read().splitlines()
    return '\n'.join(l.rstrip() for l in lines
                     if not re.search(r'it/s\]|s/it\]|Warning|warn\(|^\s*$|Loading|Fetching|Downloading|%\||^  chunk |^\$ ', l))
print(f'GPU: {torch.cuda.get_device_name(0)}  config: {CFG}')
for p in ['/content/log_ablate_report.txt']:
    print(f'===== {p.split("/")[-1]} =====')
    print(clean(p))

In [ ]:
if AUTO_DISCONNECT:
    from google.colab import runtime
    print('Done. Disconnecting the runtime to save compute units (outputs above stay visible).')
    runtime.unassign()